# Virtual Mouse / Air Canvas - Phase 2

**Threaded capture + hand-landmark display.**

This notebook opens the webcam, tracks one hand with MediaPipe, draws the
21-point skeleton, marks the index fingertip, and shows a live FPS counter.
No cursor control yet - this phase proves the capture + tracking foundation
is fast and stable.

Run the cells top to bottom. The last cell opens a live window; press **`q`**
in that window (or interrupt the kernel) to stop.

---
### Important: library versions
Newest MediaPipe (0.10.22+) removed the classic `mediapipe.solutions.hands`
API this code uses. Install the pinned versions in the next cell, or the
import will fail with `module 'mediapipe' has no attribute 'solutions'`.

## 1. Install dependencies (run once)
Uncomment and run this cell the first time. Restart the kernel afterwards.

In [1]:
# !pip install opencv-python==4.9.0.80 mediapipe==0.10.21 numpy==1.26.4 pyautogui==0.9.54

## 2. Imports and a quick version check

In [2]:
import time
import platform
import threading
from collections import deque

import cv2
import numpy as np
import mediapipe as mp

print("opencv   :", cv2.__version__)
print("numpy    :", np.__version__)
print("mediapipe:", mp.__version__)
assert hasattr(mp, "solutions") and hasattr(mp.solutions, "hands"), (
    "Wrong MediaPipe version - install mediapipe==0.10.21 (see cell 1)."
)
print("solutions.hands available - OK")

opencv   : 4.11.0
numpy    : 1.26.4
mediapipe: 0.10.21
solutions.hands available - OK


## 3. Threaded camera

A background thread grabs frames as fast as the camera produces them into a
one-slot buffer (`deque(maxlen=1)`). The main loop always reads the freshest
frame; stale frames are dropped automatically, so the slow blocking
`cap.read()` never throttles processing.

In [3]:
class ThreadedCamera:
    def __init__(self, src=0, width=640, height=480):
        if platform.system() == "Windows":
            self.cap = cv2.VideoCapture(src, cv2.CAP_DSHOW)  # fast startup on Windows
        else:
            self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        self.buffer = deque(maxlen=1)
        self.running = False
        self.lock = threading.Lock()
        self.thread = None

    def opened(self):
        return self.cap.isOpened()

    def start(self):
        self.running = True
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def _loop(self):
        while self.running:
            ok, frame = self.cap.read()
            if ok:
                with self.lock:
                    self.buffer.append(frame)

    def read(self):
        with self.lock:
            return self.buffer[-1] if self.buffer else None

    def stop(self):
        self.running = False
        if self.thread is not None and self.thread.is_alive():
            self.thread.join(timeout=1)
        self.cap.release()

## 4. Hand tracker (MediaPipe wrapper)

All MediaPipe code lives here - the vision layer. Landmark indices that matter
for later gesture phases: **4** thumb tip, **8** index tip, **12** middle tip,
**16** ring tip, **20** pinky tip.

In [4]:
WRIST, THUMB_TIP, INDEX_TIP, MIDDLE_TIP, RING_TIP, PINKY_TIP = 0, 4, 8, 12, 16, 20


class HandTracker:
    def __init__(self, max_hands=1, detection_conf=0.7, tracking_conf=0.5):
        self.mp_hands = mp.solutions.hands
        self.mp_draw = mp.solutions.drawing_utils
        self.mp_styles = mp.solutions.drawing_styles
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=max_hands,
            min_detection_confidence=detection_conf,
            min_tracking_confidence=tracking_conf,
        )

    def process(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = self.hands.process(rgb)
        rgb.flags.writeable = True
        return results

    def draw(self, frame_bgr, results):
        if results.multi_hand_landmarks:
            for lm in results.multi_hand_landmarks:
                self.mp_draw.draw_landmarks(
                    frame_bgr, lm, self.mp_hands.HAND_CONNECTIONS,
                    self.mp_styles.get_default_hand_landmarks_style(),
                    self.mp_styles.get_default_hand_connections_style(),
                )

    @staticmethod
    def landmark_pixel(hand_landmarks, index, frame_shape):
        h, w = frame_shape[:2]
        lm = hand_landmarks.landmark[index]
        return int(lm.x * w), int(lm.y * h)

    def close(self):
        self.hands.close()

## 5. Run live tracking

Opens a webcam window. Put your hand in view to see the skeleton, a green dot
on the index fingertip, and the FPS counter. **Press `q` in the window to stop.**

> Note: `cv2.imshow` opens a separate OS window, not inline in the notebook.
> If the window ever hangs, interrupt the kernel and run the cleanup cell.

In [8]:
camera = ThreadedCamera(src=0, width=640, height=480)

if not camera.opened():
    print("ERROR: could not open the camera.")
    print(" - Close any app using the webcam (Zoom, Teams, browser).")
    print(" - If you have more than one camera, try src=1.")
else:
    camera.start()
    time.sleep(1.0)               # warm up
    tracker = HandTracker(max_hands=1)
    prev = time.time()
    print("Tracking started. Press 'q' in the window to quit.")
    try:
        while True:
            frame = camera.read()
            if frame is None:
                continue
            frame = cv2.flip(frame, 1)               # mirror
            results = tracker.process(frame)
            tracker.draw(frame, results)
            if results.multi_hand_landmarks:
                for lm in results.multi_hand_landmarks:
                    cx, cy = tracker.landmark_pixel(lm, INDEX_TIP, frame.shape)
                    cv2.circle(frame, (cx, cy), 10, (0, 255, 0), cv2.FILLED)
            now = time.time()
            fps = 1.0 / (now - prev) if now != prev else 0.0
            prev = now
            cv2.putText(frame, f"FPS: {int(fps)}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.imshow("Phase 2 - Hand Tracking", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        camera.stop()
        tracker.close()
        cv2.destroyAllWindows()
        print("Stopped cleanly.")

Tracking started. Press 'q' in the window to quit.
Stopped cleanly.


## 6. Cleanup (run if the window hangs)
If the kernel was interrupted mid-loop, run this to release the camera and
close any leftover windows.

In [7]:
try:
    camera.stop()
except Exception:
    pass
cv2.destroyAllWindows()
for _ in range(4):
    cv2.waitKey(1)
print("cleaned up")

cleaned up
